# 02 — Feature Engineering Deep Dive

This notebook walks through the feature engineering pipeline including velocity features, behavioural deviation, and graph-derived features.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.feature_store import build_features, MODEL_FEATURES
from src.models.graph_features import build_graph_features

pd.set_option("display.max_columns", 50)

## Build features

In [ ]:
train = pd.read_csv("../data/synthetic/train.csv")
accounts = pd.read_csv("../data/synthetic/accounts.csv")
customers = pd.read_csv("../data/synthetic/customers.csv")

df = build_features(train, accounts, customers)
df = build_graph_features(df)
print(f"Feature matrix: {df.shape}")
print(f"\nModel features ({len(MODEL_FEATURES)}):")
for f in MODEL_FEATURES:
    print(f"  {f}")

## Velocity feature distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
velocity_cols = ["txn_count_1h", "txn_count_24h", "txn_amount_sum_1h",
                 "unique_merchants_1h", "unique_devices_24h", "unique_geos_24h"]
for ax, col in zip(axes.flat, velocity_cols):
    df[col].clip(upper=df[col].quantile(0.99)).hist(bins=30, ax=ax, color="steelblue", alpha=0.7)
    ax.set_title(col)
plt.suptitle("Velocity feature distributions", y=1.01)
plt.tight_layout()
plt.show()

## Velocity features by fraud label

In [ ]:
for col in ["txn_count_1h", "txn_count_24h", "txn_amount_sum_1h"]:
    fraud_mean = df.loc[df["label_fraud"] == 1, col].mean()
    legit_mean = df.loc[df["label_fraud"] == 0, col].mean()
    print(f"{col:30s}  fraud={fraud_mean:.2f}  legit={legit_mean:.2f}  ratio={fraud_mean/max(legit_mean,0.01):.2f}")

## Behavioural deviation features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["amount_z"].clip(-5, 5).hist(bins=50, ax=axes[0], color="steelblue", alpha=0.7)
axes[0].set_title("Amount z-score distribution")
df["peer_amount_ratio"].clip(0, 5).hist(bins=50, ax=axes[1], color="steelblue", alpha=0.7)
axes[1].set_title("Peer amount ratio distribution")
plt.tight_layout()
plt.show()

## Graph features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["graph_degree"].clip(upper=50).hist(bins=30, ax=axes[0], color="steelblue", alpha=0.7)
axes[0].set_title("Graph degree")
df["graph_neighbor_fraud_ratio"].hist(bins=30, ax=axes[1], color="salmon", alpha=0.7)
axes[1].set_title("Neighbour fraud ratio")
plt.tight_layout()
plt.show()

print(f"PageRank range: [{df['graph_pagerank'].min():.6f}, {df['graph_pagerank'].max():.6f}]")
print(f"Community fraud ratio range: [{df['graph_community_fraud_ratio'].min():.4f}, {df['graph_community_fraud_ratio'].max():.4f}]")

## Feature correlation with fraud label

In [ ]:
corr = df[MODEL_FEATURES + ["label_fraud"]].corr()["label_fraud"].drop("label_fraud").sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 8))
corr.plot.barh(ax=ax, color=["salmon" if v > 0 else "steelblue" for v in corr])
ax.set_title("Feature correlation with fraud label")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()